# Retail Customer Purchase Intelligence — Milestone 1
**Participant:** Participant_M1_04  
**Project:** Retail Customer Purchase Intelligence  
**Milestone:** 1  
**Dataset package:** `Retail.zip`

## Objective
Predict purchase amounts, segment customers, understand purchase drivers, and support marketing/inventory decisions.

## Milestone scope
join customer demographics and purchase details using User_ID; inspect quality; EDA; clean/engineer features; baseline Linear Regression; evaluate MAE, RMSE, and R².

> This is a synthetic sample submission for evaluator testing. Dataset files must be placed in the paths referenced below. Results are not pre-filled or fabricated.

## 1. Load and join data
The project brief lists demographic, purchase, and review data. Review text is kept separate unless needed for the NLP milestone.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("data")
demo_path = DATA_DIR / "user_demographics.csv"
purchase_path = DATA_DIR / "User_product_purchase_details.csv"

demo = pd.read_csv(demo_path)
purchase = pd.read_csv(purchase_path)
print("demographics:", demo.shape, "purchases:", purchase.shape)
print(demo.dtypes)
print(purchase.dtypes)

In [ ]:
df = purchase.merge(demo, on="User_ID", how="left", validate="many_to_one")
print("joined shape:", df.shape)
display(df.head())
print("missing values, top 15:")
display(df.isna().sum().sort_values(ascending=False).head(15))
print("duplicate rows:", df.duplicated().sum())

## 2. Exploratory analysis
Examine purchase distribution and differences across customer/product categories. Avoid interpreting category codes as meaningful numeric quantities.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

target = "Purchase"
print(df[target].describe())
sns.histplot(df[target], bins=40, kde=True)
plt.title("Purchase amount distribution")
plt.show()

for col in ["Gender", "Age", "City_Category", "Product_Category_1"]:
    if col in df.columns:
        display(df.groupby(col, dropna=False)[target].agg(["count", "mean", "median"]).sort_values("mean", ascending=False).head(12))

## 3. Cleaning, features, and baseline
Use a train/test split and fit preprocessing only on training data to reduce leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

target = "Purchase"
drop_cols = [target, "User_ID", "Product_ID"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = pd.to_numeric(df[target], errors="coerce")
valid = y.notna()
X, y = X.loc[valid], y.loc[valid]
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=np.number).columns.tolist()
pre = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = Pipeline([("prep", pre), ("regressor", LinearRegression())])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print({"MAE": mean_absolute_error(y_test, pred),
       "RMSE": mean_squared_error(y_test, pred) ** 0.5,
       "R2": r2_score(y_test, pred)})

## 4. Findings and limitations
- Record actual metric values after running the notebook; do not claim improvement without a comparison.
- Check whether the join caused unmatched demographics.
- User and product identifiers are excluded from the baseline features to avoid treating IDs as meaningful measurements.

## Additional review notes
The baseline workflow is included, but further work is needed to document the effect of each cleaning choice, compare against a simple baseline, and explain errors by relevant groups.